<a href="https://colab.research.google.com/github/Saideshmukh717/movie-project-for-microsoft/blob/main/movies.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

In [3]:
df = pd.read_csv('/cleaned_movies_data.csv')

display(df)

,budget,genres,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,...,vote_count,cast,crew,tags,year,engagement_score,weighted_rating,popularity_category,overview_length,success_score
0,237000000,"Action, Adventure, Fantasy, Science Fiction",19995,"culture clash, future, space war, space colony...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"Ingenious Film Partners, Twentieth Century Fox...","United States of America, United Kingdom",...,11800,"{'Jake Sully': 'Sam Worthington', 'Neytiri': '...","{'Editor': 'John Refoua', 'Production Design':...",action adventure fantasy science fiction cultu...,2009.0,84960.0,7.050468,Very High,28,48.828118
1,300000000,"Adventure, Fantasy, Action",285,"ocean, drug abuse, exotic island, east india t...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"Walt Disney Pictures, Jerry Bruckheimer Films,...",United States of America,...,4500,"{'Captain Jack Sparrow': 'Johnny Depp', 'Will ...","{'Director of Photography': 'Dariusz Wolski', ...",adventure fantasy action ocean drug abuse exot...,2007.0,31050.0,6.665426,Very High,34,45.122943
2,245000000,"Action, Adventure, Crime",206647,"spy, based on novel, secret agent, sequel, mi6...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"Columbia Pictures, Danjaq, B24","United Kingdom, United States of America",...,4466,"{'James Bond': 'Daniel Craig', 'Blofeld': 'Chr...","{'Original Music Composer': 'Thomas Newman', '...",action adventure crime spy based novel secret ...,2015.0,28135.8,6.239307,Very High,41,35.397641
3,250000000,"Action, Crime, Drama, Thriller",49026,"dc comics, crime fighter, terrorist, secret id...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"Legendary Pictures, Warner Bros., DC Entertain...",United States of America,...,9106,"{'Bruce Wayne / Batman': 'Christian Bale', 'Al...","{'Original Music Composer': 'Hans Zimmer', 'Pr...",action crime drama thriller dc comic crime fig...,2012.0,69205.6,7.346396,Very High,65,37.499515
4,260000000,"Action, Adventure, Science Fiction",49529,"based on novel, mars, medallion, space travel,...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,Walt Disney Pictures,United States of America,...,2124,"{'John Carter': 'Taylor Kitsch', 'Dejah Thoris...","{'Screenplay': 'Mark Andrews', 'Director': 'An...",action adventure science fiction based novel m...,2012.0,12956.4,6.096324,Very High,55,16.257151
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4795,220000,"Action, Crime, Thriller",9367,"united states–mexico barrier, legs, arms, pape...",es,El Mariachi,El Mariachi just wants to play his guitar and ...,14.269792,Columbia Pictures,"Mexico, United States of America",...,238,"{'El Mariachi': 'Carlos Gallardo', 'Bigotón': ...","{'Director': 'Robert Rodriguez', 'Director of ...",action crime thriller united statesmexico barr...,1992.0,1570.8,6.150226,High,62,7.359512
4796,9000,"Comedy, Romance",72766,NaN,en,Newlyweds,A newlywed couple's honeymoon is upended by th...,0.642552,NaN,NaN,...,5,"{'Buzzy': 'Edward Burns', 'Linda': 'Kerry Bish...","{'Director': 'Edward Burns', 'Producer': 'Aaro...",comedy romance newlywed couple honeymoon upend...,2011.0,29.5,6.091563,Low,13,3.238620
4797,0,"Comedy, Drama, Romance, TV Movie",231617,"date, love at first sight, narration, investig...",en,"Signed, Sealed, Delivered","""Signed, Sealed, Delivered"" introduces a dedic...",1.444476,"Front Street Pictures, Muse Entertainment Ente...",United States of America,...,6,"{'Oliver O’Toole': 'Eric Mabius', 'Shane McIne...","{'Costume Design': 'Carla Hetland', 'Producer'...",comedy drama romance tv movie date love first ...,2013.0,42.0,6.095033,Low,73,3.480946
4798,0,NaN,126186,NaN,en,Shanghai Calling,When ambitious New York attorney Sam is sent t...,0.857008,NaN,

In [4]:
indices = pd.Series(df.index,index = df['title']).drop_duplicates()
indices

,0
title,
Avatar,0
Pirates of the Caribbean: At World's End,1
Spectre,2
The Dark Knight Rises,3
John Carter,4
...,...
El Mariachi,4795
Newlyweds,4796
"Signed, Sealed, Delivered",4797


text vectorization and making matrix of vectors

In [5]:
tfidf = TfidfVectorizer(max_features=50000,ngram_range=(1,2),stop_words='english')

In [6]:
tfidf_matrix = tfidf.fit_transform(df['tags'])

In [7]:
tfidf_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 251056 stored elements and shape (4800, 50000)>

Applying ML model - Cosing Similarity

In [8]:
def recommend(title, n = 10):
  # Create a lowercase version of the indices with hyphens removed for lookup
  # This dictionary will map processed title to original df index
  p_indices = {k.replace('-', ' ').lower(): v for k, v in indices.items()}
  p_title = title.replace('-', ' ').lower()

  if p_title not in p_indices:
    return ['Movie not found']

  # Get the original dataframe index using the processed title lookup
  idx = p_indices[p_title]
  sim_score = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
  similar_idx = sim_score.argsort()[::-1][1:n+1]
  return df['title'].iloc[similar_idx]

Testing

In [18]:
recommend('the dark knight')

,title
3,The Dark Knight Rises
119,Batman Begins
428,Batman Returns
299,Batman Forever
3852,"Batman: The Dark Knight Returns, Part 2"
9,Batman v Superman: Dawn of Justice
1359,Batman
210,Batman & Robin
205,Sherlock Holmes: A Game of Shadows
2507,Slow Burn


Making pickle files for backend of website

In [19]:
pickle.dump(tfidf_matrix,open('tfidf_matrix.pkl','wb'))

pickle.dump(indices,open('indices.pkl','wb'))

df.to_pickle('df.pkl')

pickle.dump(tfidf,open('tfidf.pkl','wb'))